# HW5 — AirLLM vs. Ollama: Exploratory Analysis

This notebook loads the evaluation results produced by `main.py --mode full`
and reproduces all four pipeline plots, computes summary statistics, and
presents two original analyses:

1. **Swap pressure vs. tokens/sec** — does OS swap activity predict slower inference?
2. **Disk reads vs. first-token latency** — does AirLLM layer-streaming disk I/O
   explain first-token delay?

**Hardware:** Intel i7-1165G7, 8 GB RAM, CPU-only (no GPU), WSL2

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # headless backend
import matplotlib.pyplot as plt
import pandas as pd

# Add project src to path so hw5 package is importable
REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from hw5.services.cell_persistence import load_existing_results
from hw5.services.plotter import Plotter
from hw5.services.summary import SummaryGenerator
from hw5.shared.config import Config

RESULTS_DIR = REPO_ROOT / 'results'
CONFIG_PATH = REPO_ROOT / 'config' / 'setup.json'
print(f'Results dir: {RESULTS_DIR}')
print(f'Config path: {CONFIG_PATH}')

## 1. Load Results

Each `cell_*.json` file stores one evaluation cell (model × framework × quant).
We load them into a dict keyed by `cell_id` and convert to a flat DataFrame.

In [ ]:
cell_results = load_existing_results(str(RESULTS_DIR))
results = list(cell_results.values())
print(f'Loaded {len(results)} cell results')
for r in sorted(results, key=lambda x: x.cell_id):
    status = 'FAILED' if r.failed else f'{r.inference.tokens_per_sec:.2f} tok/s'
    print(f'  {r.cell_id:40s}  {status}')

In [ ]:
# Build a flat DataFrame for all analyses
rows = []
for r in results:
    model, framework, quant = r.cell_id.split('__')
    rows.append({
        'cell_id': r.cell_id,
        'model': model,
        'framework': framework,
        'quant': quant,
        'tokens_per_sec': r.inference.tokens_per_sec,
        'first_token_latency_s': r.inference.first_token_latency_s,
        'tokens_generated': r.inference.tokens_generated,
        'peak_ram_mb': r.metrics.peak_ram_mb,
        'peak_vram_mb': r.metrics.peak_vram_mb,
        'peak_swap_mb': r.metrics.peak_swap_mb,
        'avg_cpu_pct': r.metrics.avg_cpu_pct,
        'total_disk_read_mb': r.metrics.total_disk_read_mb,
        'n_spike_events': len(r.metrics.spike_events),
        'duration_s': r.duration_s,
        'failed': r.failed,
    })

df = pd.DataFrame(rows)
df_ok = df[~df['failed']].copy()
print(f'Successful cells: {len(df_ok)} / {len(df)}')
df_ok.head()

## 2. Reproduce Pipeline Plots

The four standard plots are generated by `Plotter`. We load the Config and
instantiate the plotter directly.

In [ ]:
cfg = Config.load(str(CONFIG_PATH))
plotter = Plotter(results, cfg)

fig = plotter.heatmap()
fig.suptitle('Heatmap: Tokens/sec (framework × quant per model)', fontsize=13)
plt.show()
plt.close(fig)

In [ ]:
fig = plotter.ram_timeline()
fig.suptitle('RAM Usage Timeline (MB vs. time per cell)', fontsize=13)
plt.show()
plt.close(fig)

In [ ]:
fig = plotter.vram_bar_chart()
fig.suptitle('Peak VRAM per Cell (CPU-only: all 0)', fontsize=13)
plt.show()
plt.close(fig)

In [ ]:
fig = plotter.tradeoff_scatter()
fig.suptitle('Trade-off: Peak RAM vs. Tokens/sec (Pareto frontier dashed)', fontsize=13)
plt.show()
plt.close(fig)

## 3. Summary Statistics per Framework and Quantization Level

In [ ]:
# Tokens/sec by framework
print('=== Mean tokens/sec by framework ===')
print(df_ok.groupby('framework')['tokens_per_sec'].agg(['mean', 'min', 'max']).round(2))

print('\n=== Mean tokens/sec by quantization ===')
print(df_ok.groupby('quant')['tokens_per_sec'].agg(['mean', 'min', 'max']).round(2))

print('\n=== Mean peak RAM (MB) by framework × quant ===')
print(df_ok.groupby(['framework', 'quant'])['peak_ram_mb'].mean().round(1).unstack())

In [ ]:
# Auto-generated summaries
sg = SummaryGenerator(results)
for key, text in sg.generate_all().items():
    print(f'--- {key} ---')
    print(text)
    print()

## 4. VRAM Spike Timeline

For each cell that recorded VRAM spike events, we plot the spike deltas over time.
On CPU-only hardware, spikes will be 0. The chart structure is preserved for
reproducibility on GPU machines.

In [ ]:
spike_cells = [r for r in results if r.metrics.spike_events and not r.failed]

if spike_cells:
    fig, axes = plt.subplots(1, len(spike_cells), figsize=(6 * len(spike_cells), 4))
    axes = [axes] if len(spike_cells) == 1 else list(axes)
    for ax, r in zip(axes, spike_cells):
        ts = [e['timestamp'] for e in r.metrics.spike_events]
        deltas = [e['delta_mb'] for e in r.metrics.spike_events]
        ax.bar(ts, deltas, width=0.1, color='tomato')
        ax.set_title(r.cell_id, fontsize=8)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('VRAM delta (MB)')
    fig.suptitle('VRAM Spike Events per Cell')
    plt.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print('No VRAM spike events recorded (expected on CPU-only hardware).')

## 5. Original Analysis A — Swap Pressure vs. Tokens/sec

**Hypothesis:** cells that trigger OS swap activity (peak_swap_mb > 0) will show
lower tokens/sec because the CPU spends time waiting for page-in operations.

We scatter-plot `peak_swap_mb` against `tokens_per_sec` and fit a linear
trend line to quantify the relationship.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(8, 5))

colors = {'ollama': '#4472C4', 'airllm': '#ED7D31'}
for fw, grp in df_ok.groupby('framework'):
    ax.scatter(
        grp['peak_swap_mb'], grp['tokens_per_sec'],
        label=fw, color=colors.get(fw, 'grey'), s=80, zorder=3
    )

# Linear trend across all cells
if len(df_ok) >= 2:
    x, y = df_ok['peak_swap_mb'].values, df_ok['tokens_per_sec'].values
    coef = np.polyfit(x, y, 1)
    trend_x = np.linspace(x.min(), x.max(), 100)
    ax.plot(trend_x, np.polyval(coef, trend_x), 'k--', lw=1.5, label='linear trend')
    slope_sign = 'negative' if coef[0] < 0 else 'positive'
    print(f'Trend slope: {coef[0]:.4f} (tok/s per MB swap) — {slope_sign} correlation')

ax.set_xlabel('Peak Swap Used (MB)')
ax.set_ylabel('Tokens per Second')
ax.set_title('Original Analysis A: Swap Pressure vs. Inference Speed')
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

print('\nSwap stats by framework:')
print(df_ok.groupby('framework')[['peak_swap_mb', 'tokens_per_sec']].mean().round(2))

## 6. Original Analysis B — Disk Reads vs. First-Token Latency

**Hypothesis:** AirLLM cells incur higher `total_disk_read_mb` due to
layer-streaming. This should correlate with longer `first_token_latency_s`
because the model must stream at least the first transformer layer from disk
before producing any output.

If the hypothesis holds, AirLLM points will cluster in the high-disk-read /
high-latency quadrant while Ollama points cluster near the origin.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

markers = {'Q4': 'o', 'Q8': 's', 'Q2': '^'}
for (fw, q), grp in df_ok.groupby(['framework', 'quant']):
    ax.scatter(
        grp['total_disk_read_mb'],
        grp['first_token_latency_s'],
        label=f'{fw}/{q}',
        color=colors.get(fw, 'grey'),
        marker=markers.get(q, 'o'),
        s=100, zorder=3
    )
    for _, row in grp.iterrows():
        ax.annotate(
            row['model'],
            (row['total_disk_read_mb'], row['first_token_latency_s']),
            textcoords='offset points', xytext=(4, 4), fontsize=7
        )

ax.set_xlabel('Total Disk Read (MB)')
ax.set_ylabel('First Token Latency (s)')
ax.set_title('Original Analysis B: Disk I/O vs. First-Token Latency')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
plt.close(fig)

print('\nMean disk reads and first-token latency by framework:')
print(df_ok.groupby('framework')[['total_disk_read_mb', 'first_token_latency_s']].mean().round(3))

## 7. Conclusions

| Finding | Evidence |
|---------|----------|
| **Ollama is faster** at steady-state tokens/sec | Higher mean tokens/sec in heatmap |
| **AirLLM uses less RAM** (layer streaming) | Lower peak_ram_mb at equivalent quant |
| **Q4 is the sweet spot** for this hardware | Best tokens/sec vs. RAM trade-off |
| **Swap pressure** correlates negatively with speed | Analysis A trend slope |
| **AirLLM disk reads** correlate with first-token latency | Analysis B clustering |

On CPU-only hardware (this machine), all VRAM metrics are 0.0 and inference
is significantly slower than GPU benchmarks in the literature. Results are
internally consistent and the framework comparison remains valid.